# Aula 2 (2026.2): Modbus TCP em CLP real e Modelo Purdue

## Objetivo da aula

Na Aula 1, a turma escreveu uma classe `ClienteModbusTCP` em JavaScript e um servidor Modbus TCP também em JavaScript, e capturou essa comunicação no Wireshark. Nesta aula o servidor deixa de ser código e passa a ser um CLP S7-1200 real, configurado no TIA Portal como `MB_SERVER`. O mesmo `cliente.js` da Aula 1, sem nenhuma mudança de código, conecta, lê e escreve registradores no CLP.

Na segunda parte da aula, a turma usa a própria bancada (notebook, switch, CLP) para aplicar o modelo Purdue: identificar em que nível cada equipamento está, desenhar zonas e condutos, e apontar onde não existe segmentação na montagem.

## Formato da aula

Um CLP S7-1200 por dupla ou por quarteto. A sala tem CLPs com duas versões diferentes da biblioteca Modbus (legada e atual, v4.0+). O roteiro trata dos dois casos no Bloco 3.

## Roteiro em blocos

- Bloco 1: recuperando o cliente.js da Aula 1.
- Bloco 2: identificando a biblioteca Modbus do CLP.
- Bloco 3: configurando o CLP como servidor Modbus TCP.
- Bloco 4: testando com hardware real e uma Watch Table.
- Bloco 5: Wireshark, análise detalhada dos pacotes.
- Bloco 6: modelo Purdue.
- Bloco 7: mapeando a bancada em zonas e condutos.

## Bloco 1: recuperando o cliente.js

Se a pasta da Aula 1 ainda estiver disponível, abra `cliente.js` de lá. É o mesmo arquivo, nenhuma linha muda hoje. Se o computador de hoje é outro, recrie o arquivo com o código abaixo.

In [ ]:
const { Socket } = require("net");
const Modbus = require("jsmodbus");

class ClienteModbusTCP {
  constructor(ip, porta) {
    this.ip = ip;
    this.porta = porta;
    this.socket = new Socket();
    this.client = new Modbus.client.TCP(this.socket);
  }

  conectar() {
    return new Promise((resolve, reject) => {
      this.socket.on("connect", resolve);
      this.socket.on("error", reject);
      this.socket.connect({ host: this.ip, port: this.porta });
    });
  }

  async lerRegistradores(enderecoInicial, quantidade) {
    const resposta = await this.client.readHoldingRegisters(enderecoInicial, quantidade);
    return resposta.response.body.valuesAsArray;
  }

  async escreverRegistradores(enderecoInicial, valores) {
    await this.client.writeMultipleRegisters(enderecoInicial, valores);
    this.client.
  }

  encerrar() {
    this.socket.end();
  }
}

module.exports = ClienteModbusTCP;


## Bloco 2: identificando a biblioteca Modbus do CLP

A Siemens mantém duas versões da instrução `MB_SERVER`, com parâmetros de nomes diferentes. A sala tem CLPs com as duas versões.

Para descobrir qual versão está disponível: na árvore de instruções do TIA Portal, categoria **Communication > Communication Processor > MODBUS TCP**, arraste `MB_SERVER` para o bloco de programa. Se o TIA Portal oferecer mais de uma versão, escolha uma; depois observe os nomes dos parâmetros do bloco inserido.

| Nomes dos parâmetros no bloco | Biblioteca |
|---|---|
| `dataBuffer`, `connectParamServer`, `ndr`, `dataRead` (minúsculas) | Legada. Siga o Bloco 3, Caminho A. |
| `MB_HOLD_REG`, `CONNECT`, `NDR`, `DR`, `DISCONNECT` (maiúsculas) | Atual, v4.0+. Siga o Bloco 3, Caminho B. |

Em caso de dúvida, clique no bloco `MB_SERVER` já inserido e abra o painel de informação no lado direito (ícone "i"). Ele mostra a versão da instrução e um link para a documentação correspondente.

## Bloco 3: configurando o CLP como servidor Modbus TCP

### Preparação (igual para as duas bibliotecas)

Confirme o IP do CLP: em **Device configuration**, clique na porta verde PROFINET do CLP, e no painel inferior abra **Properties > General > Ethernet addresses**. O IP address aparece ali.

Crie um bloco de dados global para os holding registers. No **Project tree**, clique com o botão direito em **Program blocks > Add new block > Data block (DB)**. Nomeie como `DB_HoldingRegisters`.

Dentro do DB, crie a variável:

```text
holdingRegister : Array[0..99] of Word
```

Isso reserva 100 registradores de 16 bits, endereços Modbus 0 a 99.

Em **Properties** do DB, confirme que **Optimized block access** está marcado. A Siemens recomenda isso para os parâmetros `dataBuffer` e `MB_HOLD_REG`.

Continue no Caminho A ou B, conforme identificado no Bloco 2.

### Caminho A: biblioteca legada

Fonte: documentação Siemens, Entry-ID 102020340, "Modbus/TCP with instructions MB_CLIENT and MB_SERVER".

Abra o bloco de programa principal (**Main [OB1]**) em modo de edição. Na árvore de instruções, abra **Communication > Communication Processor > MODBUS TCP**, e arraste **MB_SERVER** para dentro do OB1.

O TIA Portal pede para criar um DB de instância. Aceite o nome sugerido (por exemplo `MB_SERVER_DB`) e clique **OK**. Cada instância de `MB_SERVER` precisa do próprio DB de instância. Se houver mais de um servidor no mesmo CLP, não reaproveite o DB.

Parâmetros de entrada:

| Parâmetro | Valor | Explicação |
|---|---|---|
| `disconnect` | `FALSE` | mantém a conexão passiva aberta, aceitando o cliente |
| `connectParamServer` | a estrutura `TCON_IP_v4` (ver seção seguinte) | descreve a conexão TCP que o servidor aceita |
| `dataBuffer` | `"DB_HoldingRegisters".holdingRegister` | aponta para o bloco de holding registers criado na preparação |

Saídas, observadas durante o teste:

| Saída | Significado |
|---|---|
| `ndr` | 1 quando um cliente Modbus escreveu dado novo (function code 6 ou 16) |
| `dataRead` | 1 quando um cliente Modbus leu dado (function code 3) |
| `error` | 1 se ocorreu erro na última operação |
| `status` | código de status detalhado |

O `MB_SERVER` aceita as function codes 3 (Read Holding Registers), 6 (Write Single Register) e 16 (Write Multiple Registers) sobre a área apontada por `dataBuffer`, as mesmas usadas pelo `cliente.js`.

### Caminho B: biblioteca atual (v4.0+)

Fonte: documentação online da Siemens para a biblioteca v4.0+ (S7-1200) e v3.x+ (S7-1500). Confira os nomes exatos dos parâmetros no painel de informação do bloco (ícone "i") depois de inseri-lo, porque essa documentação muda com mais frequência que a legada.

Abra o **Main [OB1]**, vá em **Communication > Communication Processor > MODBUS TCP**, arraste **MB_SERVER** para dentro. Aceite a criação do DB de instância, como no Caminho A.

Parâmetros de entrada:

| Parâmetro | Valor | Explicação |
|---|---|---|
| `DISCONNECT` | `FALSE` | equivalente ao `disconnect` da biblioteca legada |
| `CONNECT` | a estrutura `TCON_IP_v4` (ver seção seguinte) | descreve a conexão que o servidor aceita. Neste bloco o parâmetro é InOut |
| `MB_HOLD_REG` | `"DB_HoldingRegisters".holdingRegister` | equivalente ao `dataBuffer` legado, também InOut |

Saídas:

| Saída | Significado |
|---|---|
| `NDR` | novo dado escrito pelo cliente |
| `DR` | dado lido pelo cliente |
| `ERROR` | erro na última operação |
| `STATUS` | código de status detalhado |

A biblioteca atual aceita mais function codes que a legada: 1, 2, 3, 4, 5, 6, 8, 11, 15, 16 e 23. A aula de hoje usa só leitura e escrita de holding registers (function codes 3 e 16), igual ao Caminho A.

### A estrutura TCON_IP_v4 (comum às duas bibliotecas)

Essa estrutura descreve a conexão TCP que o servidor aceita. Fonte: documentação Siemens Entry-ID 102020340, capítulo 5.

Clique com o botão direito no parâmetro `connectParamServer` (Caminho A) ou `CONNECT` (Caminho B), escolha **Define tag** ou digite o nome de uma variável nova. Use por exemplo `connServer`. No editor de DB ou na declaração da variável, defina o tipo como `TCON_IP_v4`.

Campos a preencher:

| Campo | Valor | Explicação |
|---|---|---|
| `InterfaceId` | o HW Identifier da interface PROFINET | em Device configuration, clique na porta verde, Properties > General > HW Identifier |
| `ID` | um número entre 1 e 4095, único nesse CLP | identifica essa conexão especificamente. Use 1 se for a única |
| `ConnectionType` | 11 (decimal) | fixo, código para conexão TCP |
| `ActiveEstablished` | `FALSE` | o servidor espera a conexão. Quem inicia é o cliente.js, não o CLP |
| `RemoteAddress.ADDR[1..4]` | 0.0.0.0 | endereço do parceiro remoto. Com RemotePort igual a 0, o servidor aceita qualquer cliente, então esse campo fica menos relevante |
| `RemotePort` | 0 | aceita conexão de qualquer porta de origem, já que o cliente.js conecta de uma porta escolhida pelo sistema operacional do notebook |
| `LocalPort` | 502 | porta que o CLP vai escutar, a porta padrão do Modbus TCP, a mesma usada no cliente.js e na Aula 1 |

Essa é a mesma lógica de porta já vista no material de Modbus TCP: RemotePort igual a 0 no servidor para aceitar qualquer cliente, LocalPort igual a 502.

### Compilando e colocando em RUN

Clique com o botão direito no CLP na árvore do projeto, **Download to device > Hardware and Software (only changes)**. Siga o assistente. Se pedir para colocar a CPU em STOP, aceite.

Depois do download, coloque a CPU em RUN (botão de chave verde na barra de ferramentas, ou no próprio CLP).

Confira os LEDs do CLP fisicamente: o LED RUN deve estar verde fixo, sem o LED ERROR ou MAINT piscando. Se o LED de erro estiver aceso, verifique o `status` (Caminho A) ou `STATUS` (Caminho B) do bloco `MB_SERVER` numa Watch Table antes de continuar.

## Bloco 4: testando com hardware real

### Abrindo uma Watch Table

Antes de rodar o cliente.js, deixe o TIA Portal mostrando os valores mudando em tempo real.

Na árvore do projeto, abra **Watch and force tables**, clique duas vezes em **Add new watch table**. Na coluna **Name**, adicione as quatro posições que vamos usar: `"DB_HoldingRegisters".holdingRegister[0]` até `[3]`. Clique no ícone de óculos (Monitor all) na barra de ferramentas da Watch Table. Os valores atuais aparecem na coluna **Monitor value** e se atualizam sozinhos.

### Preparando o cliente

Descubra o IP do CLP (seção anterior) e confirme que o notebook está na mesma rede, com um ping, como na Aula 1.

Crie ou edite `testar-cliente.js` com o código abaixo, substituindo `<IP do CLP>` pelo endereço real.

In [ ]:
const ClienteModbusTCP = require("./cliente.js");

async function main() {
  const cliente = new ClienteModbusTCP("<IP do CLP>", 502);

  await cliente.conectar();
  console.log("Conectado ao CLP");

  const antes = await cliente.lerRegistradores(0, 4);
  console.log("Registradores antes:", antes);

  await cliente.escreverRegistradores(0, [11, 22, 33, 44]);
  console.log("Escrita concluida");

  const depois = await cliente.lerRegistradores(0, 4);
  console.log("Registradores depois:", depois);

  cliente.encerrar();
}

main().catch((erro) => console.error("Erro:", erro));


### Rodando o teste

Rode `node testar-cliente.js`. A saída no terminal deve ser igual à da Aula 1.

Observe a Watch Table durante a execução. No instante em que `escreverRegistradores` roda, os valores `holdingRegister[0..3]` mudam de 0 (ou o que estiver lá) para 11, 22, 33, 44, em tempo real, na tela do TIA Portal. O mesmo código JavaScript da Aula 1, sem alteração de lógica, altera o estado interno de um CLP real, e isso aparece nos dois lados ao mesmo tempo: no terminal e no TIA Portal.

### Solução de problemas

| Sintoma | Causa provável |
|---|---|
| ECONNREFUSED no terminal | CLP não está em RUN, ou a porta configurada em LocalPort não é 502 |
| Conexão trava sem erro nem sucesso | notebook e CLP não estão na mesma sub-rede, ou cabo na porta errada do switch |
| ndr ou NDR nunca vira 1 na Watch Table | dataBuffer ou MB_HOLD_REG não aponta para o DB certo, ou o DB não está com acesso otimizado |
| LED ERROR aceso no CLP | verifique o status ou STATUS na Watch Table, e a tabela de erros do bloco na ajuda do TIA Portal (ícone "i") |

## Bloco 5: Wireshark, análise detalhada dos pacotes

### Capturando

Abra o Wireshark no notebook que está rodando o cliente.js, selecione a interface Ethernet conectada ao switch, inicie a captura. Rode `node testar-cliente.js` novamente. Pare a captura e filtre por:

```text
tcp.port == 502
```

A lista de pacotes deve mostrar nove linhas, na mesma ordem em que o código executa: três do handshake TCP, depois leitura, escrita e leitura de novo, cada uma com um pacote de pedido (Query) e um de resposta (Response).

| # | Origem | Destino | Protocolo | Info |
|---|---|---|---|---|
| 1 | IP do cliente | IP do CLP | TCP | SYN, Seq=0 |
| 2 | IP do CLP | IP do cliente | TCP | SYN, ACK, Seq=0, Ack=1 |
| 3 | IP do cliente | IP do CLP | TCP | ACK, Seq=1, Ack=1 |
| 4 | IP do cliente | IP do CLP | Modbus/TCP | Query, Func 3, Read Holding Registers |
| 5 | IP do CLP | IP do cliente | Modbus/TCP | Response, Func 3, Read Holding Registers |
| 6 | IP do cliente | IP do CLP | Modbus/TCP | Query, Func 16, Write Multiple Registers |
| 7 | IP do CLP | IP do cliente | Modbus/TCP | Response, Func 16, Write Multiple Registers |
| 8 | IP do cliente | IP do CLP | Modbus/TCP | Query, Func 3, Read Holding Registers |
| 9 | IP do CLP | IP do cliente | Modbus/TCP | Response, Func 3, Read Holding Registers |

### Examinando o handshake (pacotes 1 a 3)

Clique no pacote 1. No painel do meio, expanda a camada **Ethernet II**: os campos Destination e Source mostram os endereços MAC dos dois lados. Abaixo, expanda **Internet Protocol Version 4**: os campos Source Address e Destination Address mostram os IPs, e o campo Protocol mostra 6, que identifica TCP.

Expanda **Transmission Control Protocol**. Os campos Source Port e Destination Port mostram as portas envolvidas (a porta do cliente é escolhida pelo sistema operacional, a do CLP é 502). O campo Flags mostra qual bit está marcado: no pacote 1, apenas SYN; no pacote 2, SYN e ACK juntos; no pacote 3, apenas ACK. Os campos Sequence Number e Acknowledgment Number mostram os contadores que sincronizam os dois lados.

### Examinando a leitura (pacotes 4 e 5)

Clique no pacote 4. Expanda a camada **Modbus/TCP**. Dentro dela aparecem dois grupos de campos: o cabeçalho MBAP e a PDU.

Campos do MBAP:

- Transaction Identifier: número gerado pelo cliente, usado para casar o pedido com a resposta correspondente.
- Protocol Identifier: sempre 0, identifica o protocolo como Modbus.
- Length: quantidade de bytes restantes na mensagem, contando a partir do Unit Identifier.
- Unit Identifier: identifica a unidade lógica do lado do servidor, normalmente 1.

Campos da PDU, para este pacote:

- Function Code: 3, Read Holding Registers.
- Reference Number (ou Starting Address): endereço inicial, 0.
- Word Count (ou Quantity): quantidade de registradores pedidos, 4.

Clique no pacote 5, a resposta. O Transaction Identifier deve ser idêntico ao do pacote 4. Dentro da PDU aparecem: Function Code 3, Byte Count (quantidade de bytes de dados, 8, porque são 4 registradores de 2 bytes cada), e os valores em campos chamados Register 0 a Register 3.

### Examinando a escrita (pacotes 6 e 7)

Clique no pacote 6. O Function Code agora é 16. A PDU deste pedido tem mais campos que a de leitura, porque escrever exige enviar os dados junto: Starting Address, Quantity of Registers, Byte Count, e depois os valores em si, um campo Register N para cada registrador escrito. Confira que os valores mostrados são 11, 22, 33 e 44, os mesmos passados para `escreverRegistradores` no código.

Clique no pacote 7, a resposta da escrita. Ela confirma apenas o Starting Address e a Quantity of Registers, sem repetir os valores escritos.

### Confirmando no dump de bytes

Com o pacote 6 selecionado, olhe o painel inferior, o dump hexadecimal. Ao clicar em qualquer campo no painel do meio (por exemplo, no Function Code), o byte correspondente fica destacado nos dois painéis ao mesmo tempo, no hexadecimal à esquerda e no texto ASCII à direita. Isso mostra que o Function Code não é uma abstração do Wireshark: é literalmente um byte específico dentro da mensagem TCP, na posição logo depois do cabeçalho MBAP.

### O que essa análise mostra

Nenhum campo do MBAP ou da PDU está cifrado. O Transaction Identifier, o Function Code, os endereços e os valores escritos aparecem em texto claro, legíveis por qualquer software de captura. O protocolo também não inclui nenhum campo de autenticação: não há usuário, senha ou token em nenhuma parte do pacote. Qualquer dispositivo capaz de alcançar a porta 502 do CLP pode montar um pacote igual ao pacote 6 e obter o mesmo efeito.

## Bloco 6: modelo Purdue

### O que é

O modelo Purdue (Purdue Enterprise Reference Architecture, PERA) organiza a rede de uma planta industrial em níveis hierárquicos, do processo físico até a rede corporativa. Criado nos anos 90 para manufatura integrada por computador, hoje é usado como referência em segurança OT porque mostra com clareza onde a TI termina e o controle industrial começa.

| Nível | O que fica ali | Exemplo genérico | Exemplo já visto no período |
|---|---|---|---|
| Nível 0 | Processo físico | sensores, atuadores | o processo controlado pelo CLP da Aula 2 |
| Nível 1 | Controle básico | CLP, RTU | o CLP S7-1200 configurado como MB_SERVER |
| Nível 2 | Supervisão da área | IHM, SCADA de sala de controle | o SCADA visto no material de Modbus TCP |
| Nível 3 | Operações do site | MES, historian | ainda não visto na prática |
| Nível 3.5 | DMZ industrial | zona intermediária, adição da IEC 62443 | não existe hoje na bancada |
| Nível 4 | Logística e negócios do site | sistemas de TI da planta | ainda não visto na prática |
| Nível 5 | Rede corporativa | ERP, e-mail, internet | a internet usada no Bloco 0 para baixar o Node.js e o Wireshark |

### Por que isso importa para segurança

O tráfego não deveria passar direto do Nível 5 (internet, TI corporativa) para o Nível 0 ou 1 (o CLP que controla a máquina). O Nível 3.5, a DMZ industrial, existe para quebrar esse caminho direto: nada de TI se comunica diretamente com OT, tudo passa por uma zona intermediária com regras explícitas.

O caso mais conhecido de violação dessa fronteira é o Stuxnet, descoberto em 2010. O malware atingiu CLPs Siemens que controlavam centrífugas de enriquecimento de urânio em uma instalação no Irã. A rede de controle era isolada da internet, sem conexão direta. O Stuxnet chegou até lá por pendrives infectados, levados por técnicos que cruzaram, sem perceber, exatamente a fronteira entre níveis que o modelo Purdue existe para proteger. Depois de instalado, o malware alterava a lógica do CLP e, ao mesmo tempo, mostrava aos operadores leituras de painel como se tudo estivesse normal.

### O que fica na DMZ industrial (Nível 3.5)

A DMZ industrial não é um nível de controle, é uma zona de trânsito. Componentes comuns encontrados ali:

| Componente | Função |
|---|---|
| Réplica do historian | espelha dados do Nível 3 para consulta da TI, sem dar acesso direto ao historian real |
| Servidor de patches e antivírus | distribui atualizações para a rede OT sem que os equipamentos de OT precisem sair para a internet |
| Jump host (servidor de salto) | ponto único e monitorado de acesso remoto à rede OT, em vez de conexão direta de qualquer estação de TI |

A regra prática: nenhum equipamento do Nível 4 ou 5 fala diretamente com um equipamento do Nível 0, 1, 2 ou 3. Toda comunicação para de passar pela DMZ.

### Zonas e condutos (IEC 62443)

Zona: um agrupamento de ativos que compartilham o mesmo nível de exigência de segurança.

Conduto: o caminho de comunicação entre zonas, que deveria ser controlado e monitorado, não aberto.

A norma IEC 62443 também define Security Levels (SL), de 0 a 4, para expressar contra que tipo de atacante uma zona precisa resistir:

| Security Level | Tipo de ameaça contra a qual a zona resiste |
|---|---|
| SL 0 | nenhuma exigência específica |
| SL 1 | violação casual ou acidental |
| SL 2 | violação intencional, meios simples, poucos recursos, habilidade genérica |
| SL 3 | violação intencional, meios sofisticados, recursos moderados, habilidade específica em sistemas industriais |
| SL 4 | violação intencional, meios sofisticados, recursos extensos, habilidade específica, motivação alta, como um ataque patrocinado por um estado |

Uma zona de Nível 1, com CLPs controlando um processo crítico, normalmente exige um SL mais alto que uma zona de Nível 4, com sistemas de escritório da planta.

### Um exemplo fora da bancada

Considere uma estação de tratamento de água. O mapeamento em zonas poderia ficar assim:

| Zona | Nível Purdue | Equipamentos | Conduto de entrada |
|---|---|---|---|
| Processo | 0 e 1 | bombas, válvulas, CLPs de dosagem | nenhum, zona mais interna |
| Supervisão | 2 | IHM da sala de controle, SCADA local | conduto controlado a partir da zona de processo, só leitura de status para fora |
| Operações do site | 3 | historian, servidor de relatórios | conduto controlado a partir da supervisão |
| DMZ industrial | 3.5 | réplica do historian, jump host | conduto com firewall entre operações e DMZ, e outro entre DMZ e TI |
| TI da concessionária | 4 e 5 | e-mail, ERP, acesso à internet | nenhuma conexão direta com as zonas de processo, supervisão ou operações |

Esse exemplo serve de referência para o exercício do Bloco 7: cada zona tem equipamentos parecidos entre si, e cada conduto tem uma direção e, idealmente, um controle explícito.

### A bancada de hoje

Antes de desenhar o diagrama do Bloco 7, discuta com o grupo:

O CLP configurado hoje corresponde ao Nível 1.

O notebook rodando cliente.js, numa planta real, deveria operar perto do Nível 1 ou 2, como um sistema de engenharia ou de supervisão, não como um notebook qualquer da rede.

O switch é o meio físico que carrega o tráfego entre os dois.

Existe hoje alguma zona ou conduto real entre o notebook e o CLP? A resposta observada na aula é não: qualquer notebook que alcance o IP e a porta 502 do CLP consegue fazer o que o cliente.js fez no Bloco 4.

### Limitações do modelo

O modelo Purdue descreve uma hierarquia limpa, mas redes industriais reais raramente são tão organizadas. Acesso remoto de manutenção, equipamentos que enviam dados direto para serviços de nuvem, e integrações de emergência feitas sob pressão de prazo costumam criar conexões que pulam níveis inteiros. Uma parte real do trabalho de segurança OT é comparar a rede desenhada no papel com a rede que existe de fato, e encontrar exatamente esse tipo de divergência. É isso que o exercício do Bloco 7 pratica em escala pequena.

## Bloco 7: mapeando a bancada

### Exercício em grupo

Cada dupla ou quarteto desenha, em papel ou digitalmente, a montagem usada na aula:

1. Liste os equipamentos envolvidos: notebook(s), switch, CLP, e qualquer outro dispositivo na mesma rede.
2. Para cada equipamento, escreva o nível Purdue mais adequado, usando a tabela do Bloco 6.
3. Desenhe zonas: agrupe equipamentos que deveriam compartilhar o mesmo nível de confiança.
4. Desenhe condutos: as conexões entre zonas, com uma seta indicando a direção do tráfego observado no Wireshark.
5. Em cada conduto, indique se existe algum controle real (firewall, VLAN, lista de IPs permitidos) ou se é um caminho aberto.
6. Aponte pelo menos um ponto concreto onde a segmentação está frouxa ou ausente, baseado no que foi observado rodando o cliente.js e capturando no Wireshark.

### Entregável

1. O diagrama de zonas e condutos da bancada.
2. A captura Wireshark do Bloco 5, com o pacote de escrita identificado.
3. Uma resposta por grupo: o que impediria, hoje, um notebook qualquer conectado nesse switch de fazer o mesmo que o cliente.js fez?

## Fechamento

A aula trocou o servidor Modbus TCP em JavaScript da Aula 1 por um CLP real, sem alterar o código do cliente. O protocolo Modbus TCP se comportou da mesma forma nos dois casos: o Wireshark não distingue um script de um controlador industrial do outro lado da conexão. O modelo Purdue nomeia formalmente o que ficou visível na prática: não existe fronteira entre o notebook que roda o cliente e o CLP que ele controla.

O diagrama de zonas e condutos produzido hoje é o ponto de partida para a modelagem de ameaça completa, na etapa do período que trata de ataque controlado, sempre em bancada isolada.